In [1]:
# 1. Import Library
import pandas as pd
import numpy as np
import os
import nltk

In [2]:
# 2. Konfigurasi Path dan Setup
INSET_PATH = '../../../kamus/inset_final.csv'
OUTPUT_DIR = '../../outputs/SLA'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sla_lexicon_adapted.csv')
KAMUS_DIR = '../../../kamus'
LEXICON_TXT_PATH = os.path.join(KAMUS_DIR, 'sla_lexicon_adapted.txt')


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(KAMUS_DIR, exist_ok=True) 
np.random.seed(42) 

In [3]:
# 3. Fungsi Generate Rating
def generate_vader_ratings(mean_val, std_val=1.0):
    target_sum = int(round(mean_val * 10))
    
    ratings = np.random.normal(loc=mean_val, scale=std_val, size=10)
    ratings = np.round(ratings).astype(int)
    ratings = np.clip(ratings, -4, 4)
    
    current_sum = np.sum(ratings)
    diff = target_sum - current_sum
    
    if diff != 0:
        step = 1 if diff > 0 else -1
        indices = np.arange(10)
        while diff != 0:
            np.random.shuffle(indices)
            for idx in indices:
                new_val = ratings[idx] + step
                if -4 <= new_val <= 4:
                    ratings[idx] = new_val
                    diff -= step
                    if diff == 0:
                        break
                    
    return ratings.tolist()

In [4]:
# 4. Load Data Inset
df_inset = pd.read_csv(INSET_PATH)
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

print(f"[INFO] Jumlah kata di InSet: {len(df_inset)} entri")

[INFO] Jumlah kata di InSet: 9074 entri


In [5]:
# ============================================================
# KAMUS POLITIK - VERSI DATA-DRIVEN (Final)
# ============================================================

kamus_konteks_politik = {

    # ============================================================
    # 1. NETRAL / INSTITUSIONAL & PROSEDURAL (Skor 0.0)
    # ============================================================

    # --- Lembaga & Institusi ---
    'dpr': 0.0, 'ri': 0.0, 'indonesia': 0.0, 'partai': 0.0,
    'pemerintah': 0.0, 'wakil': 0.0, 'wakil rakyat': 0.0,
    'ketua': 0.0, 'anggota': 0.0, 'komisi': 0.0, 'fraksi': 0.0,
    'mpr': 0.0, 'dprd': 0.0, 'dpd': 0.0, 'dpdri': 0.0,
    'dpr_ri': 0.0, 'dprri': 0.0, 'kpu': 0.0, 'bawaslu': 0.0,
    'dkpp': 0.0, 'kpk': 0.0, 'polri': 0.0, 'tni': 0.0,
    'bpk': 0.0, 'ojk': 0.0, 'bnn': 0.0, 'bnpt': 0.0,
    'ppatk': 0.0, 'bappenas': 0.0, 'bpbd': 0.0, 'bppt': 0.0,
    'bph': 0.0, 'bpn': 0.0, 'mkd': 0.0, 'banggar': 0.0,
    'banggardpr': 0.0, 'baleg': 0.0, 'panja': 0.0, 'pansus': 0.0,
    'bamus': 0.0, 'eksekutif': 0.0, 'legislatif': 0.0,
    'legislator': 0.0, 'legislatornya': 0.0, 'majelis': 0.0,
    'institusi': 0.0, 'senayan': 0.0, 'parliament': 0.0,
    'perwakilan': 0.0, 'kepala': 0.0, 'dewan': 0.0,

    # --- Partai Politik ---
    'nasdem': 0.0, 'pdip': 0.0, 'pdiperjuangan': 0.0,
    'gerindra': 0.0, 'pkb': 0.0, 'pdi': 0.0, 'golkar': 0.0,
    'pks': 0.0, 'fraksinasdem': 0.0, 'fraksipksdprri': 0.0,

    # --- Tokoh & Jabatan ---
    'jokowi': 0.0, 'prabowo': 0.0, 'sandiaga': 0.0, 'puan': 0.0,
    'maharani': 0.0, 'menteri': 0.0, 'gubernur': 0.0, 'bupati': 0.0,
    'walikota': 0.0, 'mentri': 0.0, 'hakim': 0.0, 'politisi': 0.0,
    'pejabat': 0.0, 'pemimpin': 0.0, 'pengamat': 0.0, 'tokoh': 0.0,
    'prof': 0.0, 'profesor': 0.0, 'deputi': 0.0, 'kakanwil': 0.0,
    'kapolda': 0.0, 'polresta': 0.0, 'polres': 0.0, 'polsek': 0.0,
    'sejumlah': 0.0, 'tersebut': 0.0, 'pak': 0.0, 'mereka': 0.0,
    'kita': 0.0, 'kalian': 0.0, 'bang': 0.0, 'orang': 0.0,
    'bapak': 0.0, 'presiden': 0.0,

    # --- Wilayah & Lokasi ---
    'jakarta': 0.0, 'provinsi': 0.0, 'kabupaten': 0.0, 'daerah': 0.0,
    'desa': 0.0, 'pusat': 0.0, 'lokal': 0.0, 'nasional': 0.0,
    'nusantara': 0.0, 'nkri': 0.0, 'banten': 0.0, 'palestina': 0.0,
    'global': 0.0, 'kab': 0.0, 'jawa': 0.0, 'barat': 0.0,
    'aceh': 0.0, 'sumatera': 0.0, 'tangerang': 0.0, 'surabaya': 0.0,
    'riau': 0.0, 'negeri': 0.0, 'sumsel': 0.0, 'sumatra': 0.0,
    'sumut': 0.0, 'internasional': 0.0, 'acehtengah': 0.0,
    'magelang': 0.0, 'bojonegoro': 0.0, 'cirebon': 0.0, 'selatan': 0.0,

    # --- Proses Legislasi & Kegiatan DPR ---
    'rapat': 0.0, 'sidang': 0.0, 'paripurna': 0.0,
    'sidangparipurna': 0.0, 'sidang paripurna': 0.0,
    'rapat paripurna': 0.0, 'rapat kerja': 0.0, 'raker': 0.0,
    'rakor': 0.0, 'rdp': 0.0, 'rdpu': 0.0, 'fgd': 0.0,
    'audiensi': 0.0, 'forum': 0.0, 'musyawarah': 0.0,
    'mufakat': 0.0, 'voting': 0.0, 'interpelasi': 0.0,
    'menginterpelasi': 0.0, 'angket': 0.0, 'hak angket': 0.0,
    'hak': 0.0, 'mosi': 0.0, 'koalisi': 0.0, 'oposisi': 0.0,
    'divoting': 0.0, 'diketok': 0.0, 'palu': 0.0, 'diskusi': 0.0,
    'dialog': 0.0, 'berdialog': 0.0, 'pemilu': 0.0, 'pilpres': 0.0,
    'pemilihan': 0.0, 'persidangan': 0.0, 'legislasi': 0.0,

    # --- Dokumen & Regulasi ---
    'uu': 0.0, 'uud': 0.0, 'ruu': 0.0, 'ruupks': 0.0,
    'peraturan': 0.0, 'kebijakan': 0.0, 'keputusan': 0.0,
    'putusan': 0.0, 'undang-undang': 0.0, 'undang': 0.0,
    'pp': 0.0, 'rpp': 0.0, 'perppu': 0.0, 'apbn': 0.0,
    'apbd': 0.0, 'apbnp': 0.0, 'rapbn': 0.0, 'rka': 0.0,
    'prolegnas': 0.0, 'kuhp': 0.0, 'perda': 0.0, 'raperda': 0.0,
    'surpres': 0.0, 'nota': 0.0, 'draf': 0.0, 'naskah': 0.0,
    'surat': 0.0, 'rumusan': 0.0, 'rancangan': 0.0, 'aturan': 0.0,
    'ketentuan': 0.0, 'omnibus': 0.0, 'omnibuslaw': 0.0,
    'omnibuslawtaatprosedural': 0.0, 'ruuinisiatifdpdri': 0.0,
    'ruudaerahkepulauan': 0.0, 'pasal': 0.0,

    # --- Aktivitas & Proses Prosedural ---
    'menyampaikan': 0.0, 'disampaikan': 0.0, 'sampaikan': 0.0,
    'menyatakan': 0.0, 'menegaskan': 0.0, 'menjelaskan': 0.0,
    'mengumumkan': 0.0, 'umumkan': 0.0, 'mengajukan': 0.0,
    'diajukan': 0.0, 'mengusulkan': 0.0, 'usulan': 0.0,
    'usul': 0.0, 'menetapkan': 0.0, 'melaksanakan': 0.0,
    'dilaksanakan': 0.0, 'pelaksanaan': 0.0, 'mengawasi': 0.0,
    'pengawasan': 0.0, 'melaporkan': 0.0, 'laporan': 0.0,
    'meninjau': 0.0, 'membahas': 0.0, 'dibahas': 0.0,
    'pembahasan': 0.0, 'pembahasannya': 0.0, 'bahas': 0.0,
    'merumuskan': 0.0, 'rumuskan': 0.0, 'mengatur': 0.0,
    'diatur': 0.0, 'menyusun': 0.0, 'merevisi': 0.0,
    'direvisi': 0.0, 'revisi': 0.0, 'menginisiasi': 0.0,
    'diinisiasi': 0.0, 'membentuk': 0.0, 'pembentukan': 0.0,
    'dirancang': 0.0, 'diputuskan': 0.0, 'ditentukan': 0.0,
    'menentukan': 0.0, 'mengubah': 0.0, 'berubah': 0.0,
    'perubahan': 0.0, 'melibatkan': 0.0, 'dipimpin': 0.0,
    'memimpin': 0.0, 'disahkan': 0.0, 'mengesahkan': 0.0,
    'pengesahan': 0.0, 'sahkan': 0.0, 'disah': 0.0,
    'mensahkan': 0.0, 'majukan': 0.0, 'memajukan': 0.0,
    'ditangani': 0.0, 'menangani': 0.0, 'diagendakan': 0.0,
    'diharmonisasikan': 0.0, 'diundangkan': 0.0,
    'penandatanganan': 0.0, 'peresmian': 0.0, 'berlanjut': 0.0,
    'dilanjutkan': 0.0, 'diteruskan': 0.0, 'menyerahkan': 0.0,
    'diparipurnakan': 0.0, 'dipertanggungjawabkan': 0.0,
    'diwujudkan': 0.0, 'memastikan': 0.0, 'menyelesaikan': 0.0,
    'diterapkan': 0.0, 'strategis': 0.0, 'strategi': 0.0,
    'menargetkan': 0.0, 'ditargetkan': 0.0, 'mendengarkan': 0.0,
    'kinerja': 0.0, 'setujui': 0.0, 'menyetujui': 0.0,
    'disetujui': 0.0, 'persetujuan': 0.0, 'disepakati': 0.0,
    'lapor': 0.0, 

    # --- Aktor & Kelompok Sosial ---
    'rakyat': 0.0, 'masyarakat': 0.0, 'warga': 0.0, 'publik': 0.0,
    'netizen': 0.0, 'mahasiswa': 0.0, 'demonstran': 0.0,
    'asn': 0.0, 'honorer': 0.0, 'buruh': 0.0, 'petani': 0.0,
    'pengusaha': 0.0, 'ngo': 0.0, 'massa': 0.0, 'calon': 0.0,
    'paslon': 0.0, 'petahana': 0.0, 'caleg': 0.0, 'timses': 0.0,
    'pemilih': 0.0, 'konstituen': 0.0, 'penyandang': 0.0,
    'perempuan': 0.0, 'nitizen': 0.0, 'semua': 0.0, 'seluruh': 0.0, 'masyarakatnya': 0.0,

    # --- Isu & Sektor Publik ---
    'ekonomi': 0.0, 'perekonomian': 0.0, 'keuangan': 0.0,
    'anggaran': 0.0, 'kesejahteraan': 0.0, 'pembangunan': 0.0,
    'infrastruktur': 0.0, 'pendidikan': 0.0, 'kesehatan': 0.0,
    'pangan': 0.0, 'migas': 0.0, 'transportasi': 0.0,
    'konstruksi': 0.0, 'investasi': 0.0, 'listrik': 0.0,
    'daya': 0.0, 'gaji': 0.0, 'rp': 0.0, 'triliun': 0.0,
    'tarif': 0.0, 'komoditas': 0.0, 'ppn': 0.0, 'subsidi': 0.0,
    'tax amnesty': 0.0, 'amnesty': 0.0, 'taxamnesty': 0.0,
    'tax': 0.0, 'impor': 0.0, 'bumn': 0.0, 'bbm': 0.0,
    'umkm': 0.0, 'ham': 0.0, 'seksual': 0.0, 'tpks': 0.0,
    'pertanahan': 0.0, 'agraria': 0.0, 'penyiaran': 0.0,
    'pertembakauan': 0.0, 'tembakau': 0.0,           # was -3 di InSet
    'perpajakan': 0.0, 'pajak': 0.0, 'kebudayaan': 0.0,
    'keagamaan': 0.0, 'budaya': 0.0, 'migran': 0.0,
    'disabilitas': 0.0, 'kekerasan': 0.0, 'perbukuan': 0.0,

    # --- Konsep Politik & Hukum ---
    'hukum': 0.0, 'sistem': 0.0,                     # was -4 di InSet
    'konstitusi': 0.0, 'kewenangan': 0.0, 'otoritas': 0.0,
    'regulasi': 0.0, 'mekanisme': 0.0, 'birokrasi': 0.0,
    'aparatur': 0.0, 'otonomi': 0.0, 'legitimasi': 0.0,
    'representasi': 0.0, 'partisipasi': 0.0, 'konsensus': 0.0,
    'kepemiluan': 0.0, 'transformasi': 0.0,
    'kebijakan_publik': 0.0, 'etik': 0.0, 'persaudaraan': 0.0,
    'konsil': 0.0,

    # --- Kata Waktu & Umum ---
    'waktu': 0.0, 'tahun': 0.0, 'bulan': 0.0, 'minggu': 0.0,
    'hari': 0.0, 'pertama': 0.0, 'kedua': 0.0,
    'selanjutnya': 0.0, 'akhir': 0.0, 'masa': 0.0,
    'periode': 0.0, 'wib': 0.0, 'senin': 0.0, 'selasa': 0.0,
    'rabu': 0.0, 'kamis': 0.0, 'jumat': 0.0, 'sabtu': 0.0,
    'januari': 0.0, 'februari': 0.0, 'maret': 0.0, 'april': 0.0,
    'mei': 0.0, 'juni': 0.0, 'juli': 0.0, 'agustus': 0.0,
    'september': 0.0, 'oktober': 0.0, 'november': 0.0,
    'desember': 0.0, 'tanggal': 0.0,                 # was -4 di InSet

    # --- Kegiatan & Agenda Khusus ---
    'kunjungan kerja': 0.0, 'kunker': 0.0, 'kunkerdpr': 0.0,
    'reses': 0.0, 'dprreses': 0.0, 'resesdpra': 0.0,
    'sosialisasi': 0.0, 'sosialisasikan': 0.0,
    'silaturahim': 0.0, 'silaturahmi': 0.0, 'kunjungan': 0.0,
    'mengunjungi': 0.0, 'dikunjungi': 0.0, 'datangi': 0.0,
    'temui': 0.0, 'hadiri': 0.0, 'dihadiri': 0.0, 'menghadiri': 0.0,
    'kegiatan': 0.0, 'menggelar': 0.0,

    # --- Struktur & Proses Legislatif Lain ---
    'agenda': 0.0, 'agendadpr': 0.0, 'masukan': 0.0,
    'aspirasi': 0.0, 'aspirasinya': 0.0, 'pendapat': 0.0,
    'pandangan': 0.0, 'komentar': 0.0, 'koreksi': 0.0,
    'bantahan': 0.0, 'survei': 0.0, 'evaluasi': 0.0,
    'audit': 0.0, 'rekomendasi': 0.0, 'identifikasi': 0.0,
    'penyerapan': 0.0, 'menyerap': 0.0,              # was -3 di InSet
    'penindakan': 0.0, 'pengelolaan': 0.0, 'penerapan': 0.0,
    'pengendalian': 0.0, 'pelaporan': 0.0, 'pembatasan': 0.0,
    'perencanaan': 0.0, 'penyampaian': 0.0,
    'penyusunan': 0.0, 'penyelenggaraan': 0.0,
    'penyelenggara': 0.0, 'pertanggungjawaban': 0.0,
    'pengampunan': 0.0, 'penghapusan': 0.0,          # was -3, RUU PKS context

    # --- Lain-lain Netral ---
    'sda': 0.0, 'isu': 0.0, 'media': 0.0, 'medsos': 0.0,
    'gedung': 0.0, 'pidato': 0.0, 'aset': 0.0, 'dasar': 0.0,
    'payung': 0.0, 'program': 0.0, 'target': 0.0,
    'sektor': 0.0, 'fungsi': 0.0, 'jabatan': 0.0,
    'kursi': 0.0, 'parpol': 0.0, 'kepemimpinan': 0.0,
    'pengaturan': 0.0, 'pimpinan': 0.0, 'inisiatif': 0.0,
    'prioritas': 0.0, 'dapil': 0.0, 'republik': 0.0,
    'negara': 0.0, 'dinilai': 0.0, 'menggunakan': 0.0,
    'antidialog': 0.0, 'tribun': 0.0, 'perorangan': 0.0,
    'dinsos': 0.0, 'bantuan': 0.0, 'jajaran': 0.0,
    'berbasis': 0.0, 'tugasnya': 0.0, 'pertambahan': 0.0,
    'kemenkumham': 0.0, 'kumhampasti': 0.0, 'selesaikan': 0.0,
    'hutdpr': 0.0, 'ormas': 0.0, 'pesantren': 0.0,
    'karantina': 0.0, 'paripurnadpr': 0.0, 'rangka': 0.0,
    'kenaikan': 0.0, 'jaminan': 0.0, 'petisi': 0.0,
    'peradilan': 0.0, 'pengadilan': 0.0, 'syarat': 0.0,
    'kelayakan': 0.0, 'ekspresi': 0.0, 'pelaksana': 0.0,
    'pasar': 0.0, 'diduga': 0.0, 'menjabat': 0.0,
    'pemangku': 0.0, 'menampung': 0.0, 'pemerintahan': 0.0,
    'ite': 0.0, 'hukuman': 0.0, 'kemendagri': 0.0,
    'kemendikbud': 0.0, 'kemenkeu': 0.0, 'kemenhub': 0.0,
    'kemensos': 0.0, 'kemenpanrb': 0.0, 'kemenag': 0.0,
    'setneg': 0.0, 'sara': 0.0, 'pembubaran': 0.0,
    'membubarkan': 0.0, 'batalkan': 0.0, 'dibatalkan': 0.0,
    'tenggat': 0.0, 'pemda': 0.0, 'milisi': 0.0,
    'kebangsaan': 0.0, 'mayoritas': 0.0, 'peruntukan': 0.0,
    'kewangan': 0.0, 'pengerusi': 0.0, 'penggubalan': 0.0,
    'petisyen': 0.0, 'sokongan': 0.0, 'didengar': 0.0,
    'disapa': 0.0, 'disiapkan': 0.0, 'jawabkan': 0.0,
    'memuat': 0.0, 'luncurkan': 0.0, 'siapkan': 0.0,
    'diadakan': 0.0, 'berakhir': 0.0, 'dprlive': 0.0,
    'paripurnalive': 0.0, 'tahundpr': 0.0, 'penguasa': 0.0,
    'reformasi': 0.0, 'hati': 0.0, 'populer': 0.0,                      
    'reforma': 0.0, 'apresiasi': 0.0, 'kepada': 0.0,
    'tinjau': 0.0, 'ulang': 0.0, 'visa': 0.0, 'update': 0.0,
    'rumahku': 0.0, 'viva': 0.0, 'AU': 0.0,

    # --- Angka Romawi ---
    'i': 0.0, 'ii': 0.0, 'iii': 0.0, 'iv': 0.0, 'v': 0.0,
    'vi': 0.0, 'vii': 0.0, 'viii': 0.0, 'ix': 0.0, 'x': 0.0,
    'xi': 0.0, 'xii': 0.0,

    # ============================================================
    # 2. OVERRIDE INSET — KATA UMUM DI INSET YANG SALAH SKORNYA
    # (Identifikasi data-driven: skor InSet bertentangan tanda
    # dengan distribusi kata di tweet ground truth.)
    # ============================================================

    # Kata umum/fungsi yang dilabeli InSet positif tinggi tapi netral semantik
    'ada': 0.0,        # was +4 di InSet (kata umum)
    'sudah': 0.0,      # was +3 di InSet
    'ya': 0.0,         # was +4 di InSet (kata sapaan)
    'banyak': 0.0,     # was +3 di InSet
    'sangat': 0.0,     # was +3 di InSet (intensifier umum)
    'mau': 0.0,        # was +5 di InSet
    'punya': 0.0,      # was +3 di InSet
    'lalu': 0.0,       # was +3 di InSet (penanda waktu)
    'mati': 0.0,       # was +4 di InSet (sangat tergantung konteks)
    'tahu': 0.0,       # was +4 di InSet
    'satu': 0.0,       # was +3 di InSet (numeral)
    'kepentingan': 0.0,# was +5 di InSet (sering "kepentingan pribadi")
    'menyuarakan': 0.0,# was +3 di InSet (sering ironis di NEG context)

    # Kata netral institusional yang dilabeli InSet negatif
    'atas': 0.0,       #  -4 di InSet (preposisi)
    'lembaga': 0.0,    # was -3 di InSet (kata netral)
    'tim': 0.0,        # was -4 di InSet (kata netral)
    'badan': 0.0,      # was -3 di InSet (kata institusional)
    'pihak': 0.0,      # was -3 di InSet
    'data': 0.0,       # was -3 di InSet
    'info': 0.0,       # was -3 di InSet
    'masuk': 0.0,      # was -3 di InSet (kata umum)
    'pelantikan': 0.0, # was -3 di InSet (event netral)
    'oleh': 0.0,       # was -3 di InSet (preposisi)
    'redominasi': 0.0,   # was -3 di InSet (kata netral)
    'bertemu': 0.0,      # was -3 di InSet (kata umum)
    'diimbau': 0.0,     # was -3 di InSet (kata umum)
    'tanda': 0.0,    # was -3 di InSet (kata netral)
    'diperlukan': 0.0, # was -3 di InSet (kata umum)
    'tanda tangan': 0.0, # was -3 di InSet (kata netral)
    'livestream': 0.0,   # was -3 di InSet (kata netral)
    'akademik': 0.0,     # was -3 di InSet (kata netral)
    'cipta': 0.0,       # wa -3 di InSet (kata netral)
    'dipetakan': 0.0,     # was -3 di InSet (kata netral)
    'portofolio': 0.0,     # was -3 di InSet (kata netral)

    
    # ============================================================
    # 3. POSITIF (Progres, Dukungan, Keberhasilan)
    # ============================================================

    # --- Persetujuan & Dukungan ---
    'setuju': 0.5, 'dukung': 0.8, 'mendukung': 0.5,
    'dukungan': 0.5, 'didukung': 0.5, 'sepakat': 0.8,

    # --- Apresiasi & Pujian ---
    'apresiatif': 1.5, 'puji': 1.5, 'memuji': 1.5,
    'terima kasih': 1.0, 'syukur': 1.0, 'bangga': 1.5,
    'dibanggakan': 1.5, 'dirgahayu': 0.5, 'menyemarakkan': 0.5,
    'agungkan': 0.5, 'hargai': 0.5, 'hormati': 0.5,
    'disambut': 0.5, 'menyambut': 1.0,

    # --- Keberhasilan & Progres ---
    'berhasil': 1.8, 'sukses': 1.8, 'lulus': 1.5, 'maju': 1.0,
    'kemajuan': 1.0, 'progres': 1.0, 'percepat': 1.0,
    'mempercepat': 1.0, 'meningkat': 1.0, 'memulihkan': 1.5,
    'pemulihan': 1.5, 'dikembangkan': 1.0,
    'merealisasikan': 1.5, 'direalisasikan': 1.5,
    'pertumbuhan': 0.5, 'berkelanjutan': 1.0, 'bermanfaat': 1.5,
    'manfaat': 1.0, 'efektif': 1.2, 'efisien': 1.2,

    # --- Karakter & Nilai Positif ---
    'komitmen': 0.8, 'serius': 0.5, 'jujur': 1.0,
    'integritas': 1.5, 'amanah': 1.0, 'dedikasi': 1.0,
    'transparan': 1.5, 'akuntabel': 1.5, 'responsif': 1.0,
    'proaktif': 1.0, 'inovatif': 1.2, 'kreatif': 1.0,
    'tepat': 1.0, 'mantap': 1.0, 'hebat': 1.0,
    'kompeten': 1.2, 'tegas': 0.8, 'berani': 1.2,
    'berpihak': 1.0, 'soleh': 0.5,

    # --- Kondisi Positif ---
    'sejahtera': 1.0, 'aman': 1.0, 'stabil': 1.0,
    'stabilitas': 0.8, 'positif': 1.5, 'baik': 0.8,
    'bersih': 1.0, 'adil': 1.2, 'lancar': 1.0,
    'selamat': 1.0, 'selamatkan': 0.8, 'solid': 1.0,
    'kompak': 1.0, 'bersatu': 1.0,

    # --- Nilai Demokrasi & Kebangsaan ---
    'demokrasi': 0.8, 'kedaulatan': 1.0, 'keadilan': 1.0,
    'kepastian': 0.5, 'perlindungan': 0.8, 'dilindungi': 1.2,
    'melindungi': 1.0, 'lindungi': 1.0, 'kredibilitas': 0.8,
    'pancasila': 0.5, 'gotong_royong': 0.8, 'berkeadilan': 1.2,
    'ketahanan': 0.8, 'kedamaian': 1.0,

    # --- Perjuangan & Advokasi ---
    'berjuang': 1.0, 'memperjuangkan': 1.0,
    'diperjuangkan': 1.0, 'perjuangkan': 1.0,
    'perjuangan': 0.5, 'advokasi': 1.0,

    # --- Harapan ---
    'diharapkan': 0.5, 'harapkan': 0.5, 'mengharapkan': 0.5,
    'harapan': 0.5, 'diharap': 0.5, 'semoga': 1.0,
    'bertekad': 1.0, 'merekomendasikan': 0.8, 'lestarikan': 1.0,

    # --- DATA-DRIVEN: kata yang InSet bilang negatif tapi ternyata positif ---
    'mendorong': 0.5,   # was -4 di InSet, di data 5/5 di POS
    'didorong': 0.5,
    'dorong': 0.5,

    # --- Partisipasi ---
    'representatif': 0.5, 'amanat': 0.5, 'terpilih': 0.5,
    'terpenuhi': 0.5,

    # --- Kata Positif Lemah ---
    'meningkatkan': 0.5, 'peningkatan': 0.3, 'berpotensi': 1.0,
    'targetkan': 0.3, 'dijamin': 0.5, 'tingkatkan': 0.5,
    'kabulkan': 0.5, 'asyik': 0.5, 'nurani': 0.8,

    # --- Hashtag Positif ---
    'jokowimembangunindonesia': 0.5, 'prabowosandi': 0.3,
    'hijrahbersamajokowi': 0.5, 'satujiwabersamarakyat': 0.5,

    # ============================================================
    # 4. NEGATIF (Kritik, Skandal, Kegagalan)
    # ============================================================

    # --- Sarkasme Khusus Twitter (skor moderat) ---
    'pencitraan': -2.0, 'drama': -1.5, 'sinetron': -1.5,
    'komedi': -1.2, 'lelucon': -1.5, 'plongo': -2.5,
    'halu': -2.0, 'korban': -1.5, 'masih': -0.5, 'gila': -2.0,
    'seriusi': -0.5, 'naib': -0.5,

    # --- Kritik Tajam / Makian (skor kuat tapi tidak ekstrem) ---
    'gabecus': -3.0, 'bajingan': -3.0, 'dancok': -3.5,
    'cocot': -2.5, 'ugalan': -2.5, 'mangkrak': -2.0,
    'bedebah': -3.0, 'ngeles': -1.5, 'ngeresah': -2.0,
    'fiktif': -2.0, 'zholim': -2.5, 'zolim': -2.5,
    'kezoliman': -2.5, 'penjahat': -3.0, 'musnahkan': -2.5,
    'ngawur': -2.0, 'setan': -3.0, 'iblis': -3.0, 'gibah': -2.0,

    # --- Penundaan & Kelambatan ---
    'tunda': -1.5, 'menunda': -1.5, 'penundaan': -1.2,
    'ditunda': -1.5, 'molor': -2.0, 'lambat': -1.2,
    'mengulur': -1.5, 'mandek': -1.5,

    # --- Penolakan & Protes ---
    'tolak': -1.5, 'menolak': -1.5, 'penolakan': -1.5,
    'nolak': -1.5, 'kritik': -1.0, 'protes': -1.0,
    'demo': -0.3, 'kecam': -2.0, 'desak': -0.3, 'desakan': -0.3,
    'tekan': -1.0, 'paksa': -1.5, 'intervensi': -1.0,
    'tidak setuju': -1.5, 'rezim': -1.2, 'dikompori': -1.2,
    'tolaktaxamnesty': -1.0, 'bubarkan': -1.5, 'hambalang': -2.5,
    'tuntutan': -0.3, 'sorotan': -0.3, 'gugat': -1.0,
    'menggugat': -1.0,

    # --- Skandal & Korupsi (diturunkan dari -4.0) ---
    'korupsi': -3.0, 'koruptor': -3.0, 'suap': -3.0,
    'gratifikasi': -2.5, 'manipulasi': -2.5, 'politik uang': -3.0,
    'money politics': -3.0, 'janji palsu': -2.5, 'ingkar': -2.0,
    'fitnah': -2.5, 'hoax': -2.0, 'ngehoax': -2.0, 'sesat': -2.0,
    'kolusi': -2.5, 'nepotisme': -2.5, 'kkn': -3.0,
    'abuse of power': -2.5, 'penyalahgunaan': -2.5,
    'disalahgunakan': -2.0, 'merugikan': -1.8, 'oligarki': -2.0,
    'mafia': -2.5, 'kartel': -2.5, 'monopoli': -2.0,
    'calo': -2.5, 'cukong': -2.5, 'pungli': -2.5,
    'pelicin': -2.0, 'penipuan': -2.5, 'ditipu': -2.5,

    # --- Pelanggaran & Masalah Hukum ---
    'pelanggaran': -1.8, 'terjerat': -2.0, 'tipikor': -1.5,
    'tersangka': -1.2, 'penyadapan': 0.0,         
    'kejahatan': -2.5, 'teroris': -3.0, 'terorisme': -2.5,
    'antiteroris': -1.0, 'pemerkosaan': -3.0,
    'kasus': -1.0, 'oknum': -1.2, 'seleweng': -2.5,

    # --- Sikap & Perilaku Negatif ---
    'lemah': -1.5, 'buruk': -2.0, 'kacau': -2.0,
    'kekacauan': -2.0, 'ribut': -1.2, 'anarkis': -3.0,
    'rusuh': -2.5, 'bodoh': -3.0, 'bebal': -3.0,
    'dungu': -3.0, 'malu': -2.0, 'pembohong': -3.0,
    'tipu': -2.5, 'bohong': -2.5, 'muak': -2.5, 'kecewa': -2.0,
    'zonk': -2.0, 'sandiwara': -2.0, 'tragis': -1.5,
    'hancur': -2.5, 'rusak': -2.0, 'absen': -1.5,
    'bolos': -2.5, 'titip absen': -2.5, 'tidak serius': -2.0,
    'miris': -2.0, 'konyol': -2.5, 'walkout': -1.5,
    'walk out': -0.5, 'boikot': -1.5,
    'mosi tidak percaya': -2.5, 'mundur': -1.5, 'haram': -1.5,
    'perampok': -3.0, 'dinasti': -2.0, 'otoriter': -2.5,
    'ujaran benci': -2.5, 'intimidasi': -2.0,
    'perampasan': -2.0, 'dikebut': -0.5, 'digorok': -1.0,
    'koar': -0.3, 'zina': -1.5, 'intoleransi': -2.0,
    'gagal': -2.0, 'kegagalan': -2.0,

    # --- Kondisi Negatif ---
    'masalah': -1.0,        
    'kendala': -1.0, 'halangan': -0.5, 'meresah': -1.5,
    'kontroversi': -0.8, 'sulit': -1.0, 'takut': -1.0,
    'diabaikan': -1.5, 'dihiraukan': -1.5, 'kecewakan': -0.5,
    'khianati': -1.5, 'mengancam': -1.0, 'pesimistis': -1.0,
    'menyalahkan': -1.0, 'menakuti': -1.5, 'korbankan': -1.8,
    'terbelah': -1.5, 'menghadang': -1.0, 'hentikan': -1.0,
    'larangan': -0.8,         
    'investigasi': 0.0,      
    'diselidiki': 0.0,      

    # --- Hashtag & Slang Negatif ---
    'nasdemantimahar': -1.0, 'pkbmembelarakyat': -1.0,
    'politiktanpamahar': -1.0, 'indonesialeaks': -1.5,
    'senayangantipolapikir': -1.0, 'rejimbedebah': -2.5,
    'gantialeg': -1.5, 'sahkanruupks': -1.0,
    'akhirikekerasanseksual': -2.0, 'kitaogahdingabalin': -1.5,
    'tangkapfadlidzon': -1.5, 'temanrakyat': -0.5,
    'hariantikorupsi': -1.5, 'belumtepat': -0.8,
    'racun': -1.5, 'perusak': -1.5, 'tebusan': -1.5,
    'habisi': -1.5, 'kekakcauan': -2.0, 'pemukul': -1.0,
    'predator': -1.0, 'naif': -0.5, 'dagelanwalkout': -2.0,
    'psibersihbersihdpr': -2.0, 'bangunkandpr': -1.5,
    'bersihkandpr': -1.5, 'dprloyo': -2.0, 'dprlambat': -1.8,
    'dprkosong': -1.8, 'dprtidur': -1.8,
}

In [6]:
from collections import Counter

words = list(kamus_konteks_politik.keys())

duplicates = [word for word, count in Counter(words).items() if count > 1]

print("Kata duplikat:", duplicates)

Kata duplikat: []


In [7]:
# 6. Convert ke DataFrame
df_politik = pd.DataFrame(list(kamus_konteks_politik.items()), columns=['kata', 'skor'])

print(f"[INFO] Jumlah kata di kamus politik: {len(df_politik)} entri")

[INFO] Jumlah kata di kamus politik: 945 entri


In [8]:
# 7. Merge InSet dan Kamus Politik
duplikat = set(df_inset['kata']) & set(df_politik['kata'])
print(f"\n[INFO] Ditemukan {len(duplikat)} kata yang duplikat:")
print(f"       {sorted(list(duplikat))}")

df_final = pd.concat([df_inset, df_politik], ignore_index=True)
df_final = df_final.drop_duplicates(subset=['kata'], keep='last')

print(f"\n[INFO] Total kata setelah penggabungan: {len(df_final)} entri")


[INFO] Ditemukan 161 kata yang duplikat:
       ['absen', 'ada', 'adil', 'amanah', 'anggota', 'atas', 'badan', 'bahas', 'baik', 'bangga', 'banyak', 'bebal', 'berhasil', 'bermanfaat', 'bersatu', 'bertemu', 'bohong', 'bolos', 'bulan', 'buruk', 'data', 'demo', 'desak', 'desakan', 'dewan', 'dialog', 'dinasti', 'drama', 'dukung', 'dukungan', 'dungu', 'efektif', 'efisien', 'fitnah', 'forum', 'gagal', 'gibah', 'gila', 'gratifikasi', 'hati', 'hoax', 'hukum', 'iblis', 'info', 'ingkar', 'intimidasi', 'isu', 'jujur', 'jumat', 'kebijakan', 'kegiatan', 'kekacauan', 'kekerasan', 'kepentingan', 'keputusan', 'komedi', 'komisi', 'konsensus', 'kontroversi', 'konyol', 'korban', 'korupsi', 'kreatif', 'kritik', 'lalu', 'lambat', 'lancar', 'lapor', 'laporan', 'lelucon', 'lemah', 'lembaga', 'lulus', 'mafia', 'malu', 'manfaat', 'mantap', 'masalah', 'masuk', 'mati', 'mau', 'melaksanakan', 'membahas', 'mendorong', 'menetapkan', 'mengawasi', 'mengesahkan', 'menggugat', 'mengumumkan', 'menolak', 'menyampaikan', 

In [9]:
# 8. Adaptasi Skor
df_sla = pd.DataFrame()
df_sla['kata'] = df_final['kata']

df_sla['mean'] = (df_final['skor'] * 0.8).round(3)
df_sla['std'] = 1.0

df_sla['raw_ratings'] = df_sla['mean'].apply(lambda m: generate_vader_ratings(m))

In [10]:
# 9. Simpan File
output_txt_path = os.path.join(LEXICON_TXT_PATH)

with open(output_txt_path, 'w', encoding='utf-8') as f:
    for _, row in df_sla.iterrows():
        ratings_str = ','.join(map(str, row['raw_ratings']))
        line = f"{row['kata']}\t{row['mean']}\t{row['std']}\t{ratings_str}\n"
        f.write(line)

print(f"\n[SUKSES] Lexicon TXT tersimpan di:")
print(f"   {output_txt_path}")


[SUKSES] Lexicon TXT tersimpan di:
   ../../../kamus\sla_lexicon_adapted.txt


In [11]:
# 10. Backup CSV
df_sla[['kata', 'mean', 'std', 'raw_ratings']].to_csv(
    OUTPUT_FILE, 
    sep='\t', 
    header=False, 
    index=False
)

print(f"\n[CADANGAN] Backup CSV tersimpan di: {OUTPUT_FILE}")
print("\n[SELESAI] Lexicon siap digunakan untuk VADER Scoring!")


[CADANGAN] Backup CSV tersimpan di: ../../outputs/SLA\sla_lexicon_adapted.csv

[SELESAI] Lexicon siap digunakan untuk VADER Scoring!
